# MIRI Drug Repositioning Screen — xpert-fda-miri-v1

Finalized, author-aligned, L1000-contained FDA screen. Only FDA drugs present in the L1000 embedding table (exact SMILES match, drug_idx < 8981) are screened; the HG biological embeddings are the **author reference table** — no HG training / FDA-extension inference. Run cells in order. Environment: `conda activate arc`. Heavy steps (05 benchmark, 06 finetune, 07 predict) require a GPU.

## Step 01 — Drug list + embeddings

Filters the ChEMBL FDA table (`input/fda_drugs.csv`) to drugs whose exact canonical SMILES is in the L1000 rows of the embedding mapping (`drug_idx < 8981`), then copies the author reference HG table (workflow-owned `input/HG_drug_embeddings.npy`) to `01_drug_embed/output/all_drugs_idx2HG.npy` (byte-identical parity gate). Outputs: `01_drug_embed/output/drug_list.csv` (1,004 drugs) + `all_drugs_idx2HG.npy`.

In [ ]:
!python 01_drug_embed/drug_list.py
!python 01_drug_embed/drug_embeddings.py

## Gene mapping — authoritative DTOXS -> L1000 + verification

Builds the Entrez-verified DToxS->L1000 landmark gene map (resolves the 18 legacy-symbol landmarks such as WASHC5<-KIAA0196, JPT2<-HN1L) and gates that all 978 landmarks participate in the finetune. Run before step 02.

In [ ]:
!python _lib/verify_genes.py

## Step 02 — CM baseline transcriptome

Builds the consensus cardiomyocyte baseline from DTOXS control wells using the authoritative gene map (no lost legacy-symbol genes). Output: `02_cm_baseline/output/cm_ctl_landmark.csv`.

In [ ]:
!python 02_cm_baseline/cm_baseline.py

## Step 03 — DToxS finetune data (workflow-owned, frozen)

Owns the frozen DToxS finetune h5ad (`03_cm_finetune_data/output/dtoxs.h5ad`, 521 samples / 36 drugs), moved here from `pert_data/LINCS-LDS-1593/local/v1-frozen/` so the workflow is self-contained on its finetune data (like the v1.1/v2/v3 step-03 data prep). Not rebuilt; its fold columns are regenerated in place by step 05.

In [ ]:
!ls 03_cm_finetune_data/output/

## Step 04 — XPert pretrain (reference, workflow-owned)

Holds the author pretrained XPert base weight (`pretrain_mdmt_full_50_epoch.pth`) and the L1000 pretrain YAML that produced it — the pretrain dataset lives under `pert_data/XPert-L1000/source/`. Not re-run during screening; steps 05/06 consume the weight from here.

In [ ]:
!ls 04_xpert_pretrain/

## Step 05 — Benchmark XPert on DTOXS splits

Regenerates the unified fold columns (warm-start + cold-drug, 10 folds each, ADR 0014) on the frozen `03_cm_finetune_data/output/dtoxs.h5ad`, then finetunes and benchmarks each fold on its held-out test set (weights kept). The L1000 pretrain weight is also evaluated zero-shot (no finetuning) on all 20 fold test sets and plotted as the FIRST column (`pretrain`, amber). Per-fold values are overlaid as black dots on the mean ± std bars. Side branch: downstream steps use the step-06 model. Outputs: `05_xpert_finetune_bench/output/benchmark.csv` + `benchmark.png`.

In [ ]:
!python 05_xpert_finetune_bench/create_dtoxs_splits.py
!python 05_xpert_finetune_bench/xpert_finetune_bench.py

## Step 06 — Finetune XPert on DTOXS

Full-DToxS finetune -> the final prediction model (base weight from step 04). Idempotent (weights kept). Output: `06_xpert_finetune/output/dtoxs_dtoxs/`.

In [ ]:
!python 06_xpert_finetune/xpert_finetune.py

## Step 07 — Perturbed transcriptome + z-score

Predicts drug perturbation (deg) for the 1,004 L1000-contained FDA drugs via XPert (step-06 model), then z-score normalizes the DEG per gene across drugs. Predict indexes the precomputed UniMol array by drug index, so finetune and predict consume identical features. Outputs: `07_xpert_predict/output/xpert_fda-cm_deg.csv` + `-zs.csv`.

In [ ]:
!python 07_xpert_predict/perturb_transcriptome.py
!python 07_xpert_predict/zscore.py

## Step 08 — MIRI reference gene sets + pathways

Consumes the per-GSE DEG lists in `disease_data/{GSE}/local/deg/` and produces the integrated reference (`MIRI_up-down.json`, `MIRI_rank.json`). `input/MIRI_pathways.json` is static (curated).

In [ ]:
!python 08_miri_phenotype/miri_phenotype.py

## Step 09 — Efficacy scores

Runs the three scorers (WCS+RGES, Zhang/ssCMap, pathway RGES) and computes the z-components for all screened drugs, plus the score-distribution plots. Output: `09_miri_efficacy_scores/output/scores.csv` + PNGs.

In [ ]:
!python 09_miri_efficacy_scores/miri_efficacy_scores.py

## Step 10 — Filter

Applies the WCS<0 and >=6/8 critical-pathway filters + filter-progression pyramid plot. Outputs: `10_filter/output/wcs_filtered.csv`, `path_filtered.csv`, `pyramid-filters.png`.

In [ ]:
!python 10_filter/filter.py

## Step 11 — Integrate

Computes the integrated score over all screened drugs, ranks the filtered candidates, and writes the final report: `11_integrate/output/integrated_scores.csv`, `ranked_drugs.csv`, `report.md` + rank heatmaps.

In [ ]:
!python 11_integrate/integrate.py

## Archived report stage (not a numbered step)

The agentic bilingual per-drug report (`report/`) is archived as a non-numbered stage to keep the numbered steps identical to xpert-fda-miri-v1.1/v2/v3. Run `report/run_drug_agents.py` + `report/assemble_report.py` + `report/assemble_cn_report.py` over the `10_filter` survivors with integrated percentile > 0.5.

In [ ]:
!ls report/

## Pipeline summary

The final ranked drug list is `11_integrate/output/ranked_drugs.csv` with the deterministic report at `11_integrate/output/report.md` and the archived agentic drug report at `report/output/report.md`.

In [ ]:
import pandas as pd
ranked = pd.read_csv('11_integrate/output/ranked_drugs.csv')
print(f'{len(ranked)} drugs passed the filters')
print(ranked[['drug_name', 'integrated']].head(10).to_string(index=False))